# 中华白海豚 Click 定向滤波与多径批量提取（3500条）

本 Notebook 完整包含滤波、参数实验、批量检测、路径 WAV 导出和验收代码，不依赖项目中的额外 Python 模块。

固定预处理：

- 20 kHz Butterworth 高通，原型阶数2；
- 185–215 kHz Butterworth 带阻，原型阶数2；
- SOS形式，`sosfiltfilt`零相位处理；
- `x_raw`仅用于滤波审计，`x_detect`用于定位、评分、最终路径指标和识别WAV。

最终检测采用精确率优先参数：50 μs片段、中心±0.5 ms、100 μs保护区、最大延迟1 ms、MAD峰门限5/3、SNR≥6 dB、|NCC|≥0.55、频谱余弦≥0.70、综合评分≥0.60。

运行最后的工作流单元会处理
`D:\Project_Github\Indo-Pacific-humpback-dolphin\00_Data\Positive_Clicks_3500`
中的全部3500条WAV，并将结果写入
`D:\Project_Github\Indo-Pacific-humpback-dolphin\17_20260805\multipath_results_3500`。


## 1. 核心实现

以下单元定义全部配置、滤波、候选评分、参数实验、可视化、路径WAV导出和验收函数。候选峰门限失败时产生的备用峰仅写入诊断表，不允许进入最终选择。


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""3500 条中华白海豚 click 的定向滤波、多径调参与批量提取核心代码。"""

from __future__ import annotations

import json
import os
import warnings
from dataclasses import asdict, dataclass, replace
from functools import lru_cache
from pathlib import Path
from typing import Iterable

BASE_DIR = Path(r"D:\Project_Github\Indo-Pacific-humpback-dolphin")
os.environ.setdefault("MPLCONFIGDIR", str(BASE_DIR / ".tmp" / "matplotlib"))

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import butter, correlate, find_peaks, periodogram, sosfiltfilt
from tqdm.auto import tqdm


INPUT_DIR = BASE_DIR / "00_Data" / "Positive_Clicks_3500"
WORK_DIR = BASE_DIR / "17_20260805"
OUTPUT_DIR = WORK_DIR / "multipath_results_3500"
LEGACY_TIMES_PATH = BASE_DIR / "10_AutoCorrMultipath" / "all_pulse_times.txt"

HIGH_PASS_HZ = 20_000.0
HIGH_PASS_ORDER = 2
BANDSTOP_LOW_HZ = 185_000.0
BANDSTOP_HIGH_HZ = 215_000.0
BANDSTOP_ORDER = 2
FILTER_KIND = "Butterworth SOS + zero-phase sosfiltfilt"

AUDIT_MAIN_RADIUS_MS = 0.20
AUDIT_BACKGROUND_EDGE_MS = 1.00
AUDIT_FFT_SIZE = 4096
EPS = np.finfo(np.float64).eps


@dataclass(frozen=True)
class DetectionConfig:
    """多径检测参数。"""

    segment_us: float = 50.0
    center_search_half_width_ms: float = 0.5
    direct_guard_us: float = 100.0
    max_delay_ms: float = 2.0
    candidate_min_distance_us: float = 50.0
    peak_height_mad_factor: float = 4.0
    peak_prominence_mad_factor: float = 2.0
    min_candidate_snr_db: float = 6.0
    min_abs_ncc: float = 0.50
    min_spectral_cosine: float = 0.60
    min_accept_score: float = 0.55
    weight_energy: float = 0.20
    weight_ncc: float = 0.50
    weight_spectral: float = 0.30
    max_multipaths: int = 2

    def validate(self) -> None:
        """验证参数范围及评分权重。"""
        weights = self.weight_energy + self.weight_ncc + self.weight_spectral
        if not np.isclose(weights, 1.0):
            raise ValueError(f"评分权重之和必须为1，当前为 {weights}")
        if self.segment_us <= 0 or self.max_delay_ms <= 0:
            raise ValueError("片段长度和最大时延必须大于0")
        if self.candidate_min_distance_us < self.segment_us:
            raise ValueError("候选最小间距不能小于路径片段长度")
        if self.max_multipaths not in (1, 2):
            raise ValueError("本任务最多只允许提取两条多径")


FINAL_CONFIG = DetectionConfig()


def ms_to_samples(ms: float, sample_rate: int) -> int:
    """毫秒转换为采样点。"""
    return max(1, int(round(ms * 1e-3 * sample_rate)))


def us_to_samples(us: float, sample_rate: int) -> int:
    """微秒转换为采样点。"""
    return max(1, int(round(us * 1e-6 * sample_rate)))


def rms_value(values: np.ndarray) -> float:
    """计算均方根。"""
    return float(np.sqrt(np.mean(np.square(values)))) if len(values) else 0.0


def robust_median_mad(values: np.ndarray) -> tuple[float, float]:
    """返回中位数与经1.4826校正的MAD稳健标准差。"""
    values = np.asarray(values, dtype=np.float64)
    median = float(np.median(values))
    mad = float(np.median(np.abs(values - median)))
    return median, max(1.4826 * mad, EPS)


def moving_rms(values: np.ndarray, window_samples: int) -> np.ndarray:
    """使用固定窗口计算局部RMS。"""
    kernel = np.ones(window_samples, dtype=np.float64) / window_samples
    return np.sqrt(np.convolve(values * values, kernel, mode="same"))


@lru_cache(maxsize=8)
def design_filters(sample_rate: int) -> tuple[np.ndarray, np.ndarray]:
    """按采样率设计20 kHz高通与185–215 kHz带阻SOS。"""
    nyquist = sample_rate / 2.0
    if nyquist <= BANDSTOP_HIGH_HZ:
        raise ValueError(
            f"采样率 {sample_rate} Hz 的Nyquist频率不足以设计 "
            f"{BANDSTOP_LOW_HZ:.0f}–{BANDSTOP_HIGH_HZ:.0f} Hz带阻"
        )
    high_pass = butter(
        HIGH_PASS_ORDER,
        HIGH_PASS_HZ,
        btype="highpass",
        fs=sample_rate,
        output="sos",
    )
    band_stop = butter(
        BANDSTOP_ORDER,
        [BANDSTOP_LOW_HZ, BANDSTOP_HIGH_HZ],
        btype="bandstop",
        fs=sample_rate,
        output="sos",
    )
    return high_pass, band_stop


def filter_signal(
    values: np.ndarray,
    sample_rate: int,
    normalize: bool = True,
) -> np.ndarray:
    """执行定向滤波；审计时可关闭归一化以测量实际衰减。"""
    high_pass, band_stop = design_filters(sample_rate)
    filtered = sosfiltfilt(high_pass, values)
    filtered = sosfiltfilt(band_stop, filtered)
    filtered = filtered - np.mean(filtered)
    if normalize:
        peak = float(np.max(np.abs(filtered)))
        if peak <= EPS:
            raise ValueError("滤波后信号接近全零")
        filtered = filtered / peak
    return filtered


def _to_float_mono(data: np.ndarray) -> np.ndarray:
    """将WAV数组转换为单通道float64，保留文件内相对幅度。"""
    if data.ndim == 2:
        data = data.astype(np.float64).mean(axis=1)
    original_dtype = data.dtype
    values = data.astype(np.float64)
    if np.issubdtype(original_dtype, np.integer):
        info = np.iinfo(original_dtype)
        values /= float(max(abs(info.min), abs(info.max)))
    elif not np.issubdtype(original_dtype, np.floating):
        raise TypeError(f"不支持的WAV数据类型：{original_dtype}")
    if not np.all(np.isfinite(values)):
        raise ValueError("WAV包含NaN或无穷值")
    return values


def prepare_signals(path: Path) -> tuple[int, np.ndarray, np.ndarray]:
    """读取WAV并返回审计用原始副本与检测/识别用滤波副本。"""
    sample_rate, data = wavfile.read(path)
    raw = _to_float_mono(data)
    if len(raw) < 64:
        raise ValueError(f"WAV过短，仅有 {len(raw)} 个采样点")
    raw = raw - np.mean(raw)
    raw_peak = float(np.max(np.abs(raw)))
    if raw_peak <= EPS:
        raise ValueError("WAV接近全零")
    x_raw = raw / raw_peak

    x_detect = filter_signal(x_raw, int(sample_rate), normalize=True)
    return int(sample_rate), x_raw, x_detect


def extract_centered_segment(
    values: np.ndarray,
    center_sample: int,
    segment_samples: int,
) -> tuple[np.ndarray, int, int]:
    """按中心采样点截取固定长度片段。"""
    left = segment_samples // 2
    start = center_sample - left
    end = start + segment_samples
    if start < 0 or end > len(values):
        raise ValueError(
            f"路径片段越界：center={center_sample}, start={start}, "
            f"end={end}, length={len(values)}"
        )
    return values[start:end].copy(), start, end


def find_main_arrival(
    rms_curve: np.ndarray,
    sample_rate: int,
    segment_samples: int,
    config: DetectionConfig,
) -> int:
    """在音频中心限定区域寻找局部RMS最大的主到达。"""
    midpoint = len(rms_curve) // 2
    half_width = ms_to_samples(config.center_search_half_width_ms, sample_rate)
    lower = max(segment_samples // 2, midpoint - half_width)
    upper = min(
        len(rms_curve) - (segment_samples - segment_samples // 2),
        midpoint + half_width + 1,
    )
    if lower >= upper:
        raise ValueError("中心搜索范围为空")
    return int(lower + np.argmax(rms_curve[lower:upper]))


def estimate_noise_rms(
    rms_curve: np.ndarray,
    main_center: int,
    exclusion_samples: int,
) -> float:
    """使用主到达及搜索区外的局部RMS估计稳健噪声底。"""
    mask = np.ones(len(rms_curve), dtype=bool)
    start = max(0, main_center - exclusion_samples)
    end = min(len(rms_curve), main_center + exclusion_samples + 1)
    mask[start:end] = False
    background = rms_curve[mask]
    if not len(background):
        background = rms_curve
    median, sigma = robust_median_mad(background)
    return max(median + sigma, EPS)


def max_abs_normalized_cross_correlation(
    reference: np.ndarray,
    candidate: np.ndarray,
) -> float:
    """计算允许极性翻转与微小时移的最大绝对归一化互相关。"""
    ref = reference - np.mean(reference)
    cand = candidate - np.mean(candidate)
    denominator = np.linalg.norm(ref) * np.linalg.norm(cand)
    if denominator <= EPS:
        return 0.0
    result = correlate(cand, ref, mode="full", method="auto") / denominator
    return float(np.clip(np.max(np.abs(result)), 0.0, 1.0))


def spectral_cosine_similarity(
    reference: np.ndarray,
    candidate: np.ndarray,
) -> float:
    """计算Hanning窗幅度谱的余弦相似度。"""
    window = np.hanning(len(reference))
    ref_spectrum = np.abs(np.fft.rfft(reference * window))
    cand_spectrum = np.abs(np.fft.rfft(candidate * window))
    denominator = np.linalg.norm(ref_spectrum) * np.linalg.norm(cand_spectrum)
    if denominator <= EPS:
        return 0.0
    return float(np.clip(np.dot(ref_spectrum, cand_spectrum) / denominator, 0.0, 1.0))


def _rejection_reason(
    source: str,
    snr_db: float,
    abs_ncc: float,
    spectral_cosine: float,
    score: float,
    config: DetectionConfig,
) -> str:
    """生成候选未通过原因。"""
    reasons: list[str] = []
    if source != "mad_peak":
        reasons.append("fallback_diagnostic_only")
    if snr_db < config.min_candidate_snr_db:
        reasons.append("snr_below_threshold")
    if abs_ncc < config.min_abs_ncc:
        reasons.append("ncc_below_threshold")
    if spectral_cosine < config.min_spectral_cosine:
        reasons.append("spectral_below_threshold")
    if score < config.min_accept_score:
        reasons.append("score_below_threshold")
    return ";".join(reasons)


def generate_candidates(
    x_detect: np.ndarray,
    rms_curve: np.ndarray,
    sample_rate: int,
    main_center: int,
    main_segment: np.ndarray,
    segment_samples: int,
    config: DetectionConfig,
) -> pd.DataFrame:
    """生成、评分并标记主到达之后的多径候选。"""
    guard_samples = us_to_samples(config.direct_guard_us, sample_rate)
    max_delay_samples = ms_to_samples(config.max_delay_ms, sample_rate)
    min_distance = us_to_samples(config.candidate_min_distance_us, sample_rate)
    left_margin = segment_samples // 2
    right_margin = segment_samples - left_margin
    search_start = max(main_center + guard_samples, left_margin)
    search_end = min(main_center + max_delay_samples, len(x_detect) - right_margin)
    columns = [
        "center_sample", "center_time_ms", "relative_delay_ms", "start_sample",
        "end_sample", "rms", "relative_rms_db", "energy_score", "abs_ncc",
        "spectral_cosine", "snr_db", "score", "candidate_source",
        "height_threshold", "prominence_threshold", "passes_thresholds",
        "rejection_reason",
    ]
    if search_start >= search_end:
        return pd.DataFrame(columns=columns)

    search_rms = rms_curve[search_start : search_end + 1]
    median, sigma = robust_median_mad(search_rms)
    height_threshold = median + config.peak_height_mad_factor * sigma
    prominence_threshold = config.peak_prominence_mad_factor * sigma
    local_peaks, _ = find_peaks(
        search_rms,
        height=height_threshold,
        prominence=prominence_threshold,
        distance=max(1, min_distance),
    )
    sources = ["mad_peak"] * len(local_peaks)
    if not len(local_peaks):
        fallback, _ = find_peaks(search_rms, distance=max(1, min_distance))
        if len(fallback):
            order = np.argsort(search_rms[fallback])[::-1][:10]
            local_peaks = fallback[order]
            sources = ["fallback"] * len(local_peaks)

    main_rms = rms_value(main_segment)
    noise_rms = estimate_noise_rms(
        rms_curve,
        main_center,
        max(max_delay_samples, segment_samples),
    )
    rows: list[dict[str, object]] = []
    for local_peak, source in zip(local_peaks, sources):
        center = int(search_start + local_peak)
        segment, start, end = extract_centered_segment(
            x_detect, center, segment_samples
        )
        candidate_rms = rms_value(segment)
        relative_rms_db = 20.0 * np.log10(max(candidate_rms, EPS) / max(main_rms, EPS))
        snr_db = 20.0 * np.log10(max(candidate_rms, EPS) / max(noise_rms, EPS))
        abs_ncc = max_abs_normalized_cross_correlation(main_segment, segment)
        spectral_cosine = spectral_cosine_similarity(main_segment, segment)
        energy_score = float(np.clip(candidate_rms / max(main_rms, EPS), 0.0, 1.0))
        score = (
            config.weight_energy * energy_score
            + config.weight_ncc * abs_ncc
            + config.weight_spectral * spectral_cosine
        )
        reason = _rejection_reason(
            source, snr_db, abs_ncc, spectral_cosine, score, config
        )
        rows.append(
            {
                "center_sample": center,
                "center_time_ms": 1000.0 * center / sample_rate,
                "relative_delay_ms": 1000.0 * (center - main_center) / sample_rate,
                "start_sample": start,
                "end_sample": end,
                "rms": candidate_rms,
                "relative_rms_db": relative_rms_db,
                "energy_score": energy_score,
                "abs_ncc": abs_ncc,
                "spectral_cosine": spectral_cosine,
                "snr_db": snr_db,
                "score": score,
                "candidate_source": source,
                "height_threshold": height_threshold,
                "prominence_threshold": prominence_threshold,
                "passes_thresholds": not bool(reason),
                "rejection_reason": reason,
            }
        )
    return pd.DataFrame(rows, columns=columns)


def greedy_select_candidates(
    candidates: pd.DataFrame,
    segment_samples: int,
    config: DetectionConfig,
) -> pd.DataFrame:
    """按综合评分贪婪选择最多两条不重叠多径。"""
    if candidates.empty:
        return candidates.copy()
    eligible = candidates[candidates["passes_thresholds"]].sort_values(
        "score", ascending=False
    )
    rows: list[pd.Series] = []
    centers: list[int] = []
    for _, row in eligible.iterrows():
        center = int(row["center_sample"])
        if any(abs(center - previous) < segment_samples for previous in centers):
            continue
        rows.append(row)
        centers.append(center)
        if len(rows) >= config.max_multipaths:
            break
    if not rows:
        return eligible.iloc[0:0].copy()
    return pd.DataFrame(rows).sort_values("relative_delay_ms").reset_index(drop=True)


def detect_paths(
    x_detect: np.ndarray,
    sample_rate: int,
    config: DetectionConfig,
) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    """对单个滤波信号执行主到达与多径提取。"""
    config.validate()
    segment_samples = us_to_samples(config.segment_us, sample_rate)
    if segment_samples < 3:
        raise ValueError("路径片段不足3个采样点")
    rms_curve = moving_rms(x_detect, segment_samples)
    main_center = find_main_arrival(rms_curve, sample_rate, segment_samples, config)
    main_segment, main_start, main_end = extract_centered_segment(
        x_detect, main_center, segment_samples
    )
    candidates = generate_candidates(
        x_detect,
        rms_curve,
        sample_rate,
        main_center,
        main_segment,
        segment_samples,
        config,
    )
    selected = greedy_select_candidates(candidates, segment_samples, config)
    noise_rms = estimate_noise_rms(
        rms_curve,
        main_center,
        max(ms_to_samples(config.max_delay_ms, sample_rate), segment_samples),
    )
    main_rms = rms_value(main_segment)
    rows: list[dict[str, object]] = [
        {
            "label": "main_arrival",
            "center_sample": main_center,
            "center_time_ms": 1000.0 * main_center / sample_rate,
            "relative_delay_ms": 0.0,
            "start_sample": main_start,
            "end_sample": main_end,
            "rms": main_rms,
            "relative_rms_db": 0.0,
            "energy_score": 1.0,
            "abs_ncc": 1.0,
            "spectral_cosine": 1.0,
            "snr_db": 20.0 * np.log10(max(main_rms, EPS) / max(noise_rms, EPS)),
            "score": 1.0,
        }
    ]
    for index, (_, row) in enumerate(selected.iterrows(), start=1):
        rows.append(
            {
                "label": f"multipath_{index}",
                "center_sample": int(row["center_sample"]),
                "center_time_ms": float(row["center_time_ms"]),
                "relative_delay_ms": float(row["relative_delay_ms"]),
                "start_sample": int(row["start_sample"]),
                "end_sample": int(row["end_sample"]),
                "rms": float(row["rms"]),
                "relative_rms_db": float(row["relative_rms_db"]),
                "energy_score": float(row["energy_score"]),
                "abs_ncc": float(row["abs_ncc"]),
                "spectral_cosine": float(row["spectral_cosine"]),
                "snr_db": float(row["snr_db"]),
                "score": float(row["score"]),
            }
        )
    path_table = pd.DataFrame(rows)
    if not candidates.empty:
        labels = {
            int(row["center_sample"]): f"multipath_{idx}"
            for idx, (_, row) in enumerate(selected.iterrows(), start=1)
        }
        candidates = candidates.copy()
        candidates["selected"] = candidates["center_sample"].map(labels).notna()
        candidates["selected_label"] = candidates["center_sample"].map(labels).fillna("")
    else:
        candidates = candidates.copy()
        candidates["selected"] = pd.Series(dtype=bool)
        candidates["selected_label"] = pd.Series(dtype=str)
    return path_table, candidates, rms_curve


def _periodogram_metrics(values: np.ndarray, sample_rate: int) -> dict[str, float]:
    """计算低频及200 kHz干扰审计指标。"""
    frequencies, spectrum = periodogram(
        values,
        fs=sample_rate,
        window="hann",
        nfft=AUDIT_FFT_SIZE,
        detrend="constant",
        scaling="spectrum",
    )
    valid = (frequencies >= 5_000.0) & (frequencies <= 260_000.0)
    total = float(np.sum(spectrum[valid])) + EPS
    peak_hz = float(frequencies[valid][np.argmax(spectrum[valid])])
    low_ratio = float(
        np.sum(spectrum[(frequencies >= 5_000.0) & (frequencies < HIGH_PASS_HZ)])
        / total
    )
    band_energy = float(
        np.sum(
            spectrum[
                (frequencies >= BANDSTOP_LOW_HZ)
                & (frequencies <= BANDSTOP_HIGH_HZ)
            ]
        )
    )
    band_ratio = band_energy / total
    return {
        "peak_frequency_hz": peak_hz,
        "low_frequency_ratio": low_ratio,
        "band_185_215_ratio": band_ratio,
        "total_spectrum_energy": total,
        "band_185_215_energy": band_energy,
    }


def _spectrum_cosine_band(
    raw_segment: np.ndarray,
    detect_segment: np.ndarray,
    sample_rate: int,
) -> float:
    """计算30–180 kHz主体频谱的滤波前后相似度。"""
    raw_spectrum = np.abs(
        np.fft.rfft(raw_segment * np.hanning(len(raw_segment)), AUDIT_FFT_SIZE)
    )
    detect_spectrum = np.abs(
        np.fft.rfft(detect_segment * np.hanning(len(detect_segment)), AUDIT_FFT_SIZE)
    )
    frequencies = np.fft.rfftfreq(AUDIT_FFT_SIZE, d=1.0 / sample_rate)
    keep = (frequencies >= 30_000.0) & (frequencies <= 180_000.0)
    denominator = np.linalg.norm(raw_spectrum[keep]) * np.linalg.norm(detect_spectrum[keep])
    if denominator <= EPS:
        return 0.0
    return float(np.clip(np.dot(raw_spectrum[keep], detect_spectrum[keep]) / denominator, 0.0, 1.0))


def filter_audit_row(path: Path) -> dict[str, object]:
    """计算单文件滤波前后审计指标。"""
    sample_rate, x_raw, x_detect = prepare_signals(path)
    # 使用与x_raw相同的文件级幅度标尺计算实际滤波衰减；检测仍使用归一化x_detect。
    x_detect_absolute = filter_signal(x_raw, sample_rate, normalize=False)
    audit_config = replace(
        FINAL_CONFIG,
        segment_us=50.0,
        center_search_half_width_ms=1.0,
        candidate_min_distance_us=50.0,
    )
    segment_samples = us_to_samples(audit_config.segment_us, sample_rate)
    raw_rms = moving_rms(x_raw, segment_samples)
    detect_rms = moving_rms(x_detect, segment_samples)
    raw_center = find_main_arrival(raw_rms, sample_rate, segment_samples, audit_config)
    detect_center = find_main_arrival(
        detect_rms, sample_rate, segment_samples, audit_config
    )
    radius = ms_to_samples(AUDIT_MAIN_RADIUS_MS, sample_rate)
    raw_main = x_raw[max(0, raw_center - radius) : min(len(x_raw), raw_center + radius + 1)]
    detect_main = x_detect[
        max(0, detect_center - radius) : min(len(x_detect), detect_center + radius + 1)
    ]
    common = min(len(raw_main), len(detect_main))
    edge = ms_to_samples(AUDIT_BACKGROUND_EDGE_MS, sample_rate)
    raw_background = np.concatenate([x_raw[:edge], x_raw[-edge:]])
    detect_background = np.concatenate(
        [x_detect_absolute[:edge], x_detect_absolute[-edge:]]
    )
    raw_main_metrics = _periodogram_metrics(raw_main, sample_rate)
    raw_background_metrics = _periodogram_metrics(raw_background, sample_rate)
    detect_background_metrics = _periodogram_metrics(detect_background, sample_rate)
    raw_background_rms = rms_value(raw_background)
    detect_background_rms = rms_value(detect_background)
    relative = path.relative_to(INPUT_DIR)
    return {
        "source_path": str(path.resolve()),
        "relative_path": str(relative),
        "voyage": relative.parts[0],
        "pulse_train": path.parent.parent.name,
        "click": path.parent.name,
        "sample_rate": sample_rate,
        "sample_count": len(x_raw),
        "raw_main_center_sample": raw_center,
        "detect_main_center_sample": detect_center,
        "main_shift_us": 1e6 * (detect_center - raw_center) / sample_rate,
        "raw_background_rms": raw_background_rms,
        "detect_background_rms": detect_background_rms,
        "background_rms_change_db": 20.0
        * np.log10(max(detect_background_rms, EPS) / max(raw_background_rms, EPS)),
        "raw_background_low_ratio": raw_background_metrics["low_frequency_ratio"],
        "raw_background_185_215_ratio": raw_background_metrics["band_185_215_ratio"],
        "detect_background_185_215_ratio": detect_background_metrics["band_185_215_ratio"],
        "band_185_215_attenuation_db": 10.0
        * np.log10(
            max(detect_background_metrics["band_185_215_energy"], EPS)
            / max(raw_background_metrics["band_185_215_energy"], EPS)
        ),
        "raw_background_peak_hz": raw_background_metrics["peak_frequency_hz"],
        "raw_main_185_215_ratio": raw_main_metrics["band_185_215_ratio"],
        "raw_main_peak_hz": raw_main_metrics["peak_frequency_hz"],
        "main_spectrum_cosine_30_180": _spectrum_cosine_band(
            raw_main[:common], detect_main[:common], sample_rate
        ),
    }


def scan_filter_audit(files: list[Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """全量扫描滤波审计指标，并返回错误表。"""
    rows: list[dict[str, object]] = []
    errors: list[dict[str, str]] = []
    for path in tqdm(files, desc="滤波审计", unit="wav", dynamic_ncols=True):
        try:
            rows.append(filter_audit_row(path))
        except Exception as exc:  # noqa: BLE001 - 逐文件容错是批处理要求
            warnings.warn(f"滤波审计失败，跳过 {path}: {exc}")
            errors.append({"source_path": str(path.resolve()), "stage": "filter_audit", "error": repr(exc)})
    return pd.DataFrame(rows), pd.DataFrame(errors, columns=["source_path", "stage", "error"])


def build_review_set(audit: pd.DataFrame, total: int = 200) -> pd.DataFrame:
    """构建包含200 kHz、主到达移动和航次分层的固定审查集。"""
    if len(audit) < total:
        raise ValueError(f"审计记录不足 {total} 条")
    selected: list[int] = []

    main_interference = audit[
        audit["raw_main_peak_hz"].between(BANDSTOP_LOW_HZ, BANDSTOP_HIGH_HZ)
        & (audit["raw_main_185_215_ratio"] >= 0.35)
    ].sort_values("raw_main_185_215_ratio", ascending=False)
    selected.extend(main_interference.index.tolist())

    remaining = audit.drop(index=set(selected), errors="ignore")
    selected.extend(
        remaining.nlargest(50, "raw_background_185_215_ratio").index.tolist()
    )
    remaining = audit.drop(index=set(selected), errors="ignore")
    remaining = remaining.assign(abs_main_shift=remaining["main_shift_us"].abs())
    selected.extend(remaining.nlargest(50, "abs_main_shift").index.tolist())

    selected = list(dict.fromkeys(selected))[:total]
    remaining = audit.drop(index=set(selected), errors="ignore").copy()
    rng = np.random.default_rng(20260805)
    remaining["random_order"] = rng.random(len(remaining))
    remaining = remaining.sort_values(["voyage", "random_order"])
    voyage_groups = {key: group.index.tolist() for key, group in remaining.groupby("voyage")}
    while len(selected) < total and any(voyage_groups.values()):
        for voyage in sorted(voyage_groups):
            if voyage_groups[voyage] and len(selected) < total:
                selected.append(voyage_groups[voyage].pop(0))
    result = audit.loc[selected].copy()
    result["review_order"] = np.arange(1, len(result) + 1)
    result["review_reason"] = "stratified_fill"
    main_idx = set(main_interference.index)
    result.loc[
        result["raw_background_185_215_ratio"].rank(method="first", ascending=False) <= 50,
        "review_reason",
    ] = "strong_background_200khz"
    result.loc[
        result["main_shift_us"].abs().rank(method="first", ascending=False) <= 50,
        "review_reason",
    ] = "large_filter_main_shift"
    # 主到达疑似人工定位脉冲优先级最高，不能被其他审查原因覆盖。
    result.loc[result.index.isin(main_idx), "review_reason"] = "main_200khz_interference"
    return result.sort_values("review_order").reset_index(drop=True)


def load_legacy_delays(path: Path = LEGACY_TIMES_PATH) -> dict[tuple[str, str], list[float]]:
    """读取旧ACF结果，作为非真值的一致性参考。"""
    mapping: dict[tuple[str, str], list[float]] = {}
    if not path.exists():
        return mapping
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line or line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) < 3:
            continue
        direct_ms = float(parts[2])
        mapping[(parts[0], parts[1])] = [float(value) - direct_ms for value in parts[3:]]
    return mapping


def _match_delay_counts(new: list[float], old: list[float], tolerance_ms: float = 0.10) -> int:
    """一对一匹配新旧多径延迟，仅用于参考统计。"""
    used: set[int] = set()
    matched = 0
    for value in new:
        choices = [(abs(value - old_value), index) for index, old_value in enumerate(old) if index not in used]
        if choices:
            distance, index = min(choices)
            if distance <= tolerance_ms:
                matched += 1
                used.add(index)
    return matched


def sweep_configs(base: DetectionConfig = FINAL_CONFIG) -> list[tuple[str, DetectionConfig]]:
    """生成按阶段比较的有限参数集合，避免无解释的全组合搜索。"""
    configs: list[tuple[str, DetectionConfig]] = [("final_candidate", base)]
    for segment in (40.0, 50.0, 75.0, 100.0):
        for center_width in (0.5, 1.0, 1.5):
            configs.append(
                (
                    f"geometry_seg{segment:g}_center{center_width:g}",
                    replace(
                        base,
                        segment_us=segment,
                        center_search_half_width_ms=center_width,
                        candidate_min_distance_us=max(segment, 50.0),
                    ),
                )
            )
    for guard in (75.0, 100.0, 150.0):
        for max_delay in (1.0, 2.0, 3.0, 4.0):
            configs.append(
                (
                    f"range_guard{guard:g}_delay{max_delay:g}",
                    replace(base, direct_guard_us=guard, max_delay_ms=max_delay),
                )
            )
    gate_options = [
        (3.0, 1.5, 2.0, 0.35, 0.40, 0.40),
        (3.0, 1.5, 4.0, 0.45, 0.50, 0.45),
        (4.0, 2.0, 4.0, 0.45, 0.55, 0.50),
        (4.0, 2.0, 6.0, 0.50, 0.60, 0.55),
        (5.0, 3.0, 4.0, 0.50, 0.60, 0.55),
        (5.0, 3.0, 6.0, 0.55, 0.70, 0.60),
    ]
    for index, (height, prominence, snr, ncc, spectral, score) in enumerate(gate_options, start=1):
        configs.append(
            (
                f"gates_{index}",
                replace(
                    base,
                    peak_height_mad_factor=height,
                    peak_prominence_mad_factor=prominence,
                    min_candidate_snr_db=snr,
                    min_abs_ncc=ncc,
                    min_spectral_cosine=spectral,
                    min_accept_score=score,
                ),
            )
        )
    for name, weights in (
        ("weights_20_50_30", (0.20, 0.50, 0.30)),
        ("weights_10_60_30", (0.10, 0.60, 0.30)),
        ("weights_20_60_20", (0.20, 0.60, 0.20)),
    ):
        configs.append(
            (
                name,
                replace(
                    base,
                    weight_energy=weights[0],
                    weight_ncc=weights[1],
                    weight_spectral=weights[2],
                ),
            )
        )
    unique: dict[str, DetectionConfig] = {}
    for name, config in configs:
        unique[name] = config
    return list(unique.items())


def run_parameter_sweep(review_set: pd.DataFrame) -> pd.DataFrame:
    """在固定200条审查集上运行阶段式参数比较。"""
    cached: dict[str, tuple[int, np.ndarray]] = {}
    for source in tqdm(review_set["source_path"], desc="缓存审查信号", unit="wav", dynamic_ncols=True):
        sample_rate, _, x_detect = prepare_signals(Path(source))
        cached[source] = (sample_rate, x_detect)
    legacy = load_legacy_delays()
    rows: list[dict[str, object]] = []
    for name, config in tqdm(sweep_configs(), desc="参数实验", unit="config", dynamic_ncols=True):
        counts = {0: 0, 1: 0, 2: 0}
        scores: list[float] = []
        delays: list[float] = []
        new_total = 0
        old_total = 0
        matched = 0
        boundary_main = 0
        failures = 0
        for source, pulse_train, click in review_set[["source_path", "pulse_train", "click"]].itertuples(index=False):
            try:
                sample_rate, x_detect = cached[source]
                paths, _, _ = detect_paths(x_detect, sample_rate, config)
                multipaths = paths[paths["label"] != "main_arrival"]
                count = len(multipaths)
                counts[count] += 1
                new_delays = multipaths["relative_delay_ms"].tolist()
                old_delays = legacy.get((pulse_train, click), [])
                new_total += len(new_delays)
                old_total += len(old_delays)
                matched += _match_delay_counts(new_delays, old_delays)
                scores.extend(multipaths["score"].tolist())
                delays.extend(new_delays)
                center = int(paths.iloc[0]["center_sample"])
                midpoint = len(x_detect) // 2
                half = ms_to_samples(config.center_search_half_width_ms, sample_rate)
                if abs(center - midpoint) >= max(0, half - 1):
                    boundary_main += 1
            except Exception:  # noqa: BLE001 - 实验表单独记录失败数
                failures += 1
        row: dict[str, object] = {
            "experiment": name,
            **asdict(config),
            "review_files": len(review_set),
            "failures": failures,
            "files_with_0_multipath": counts[0],
            "files_with_1_multipath": counts[1],
            "files_with_2_multipaths": counts[2],
            "selected_total": new_total,
            "mean_multipaths_per_file": new_total / max(len(review_set) - failures, 1),
            "median_selected_score": float(np.median(scores)) if scores else np.nan,
            "median_selected_delay_ms": float(np.median(delays)) if delays else np.nan,
            "legacy_total": old_total,
            "legacy_matched": matched,
            "legacy_precision_proxy": matched / max(new_total, 1),
            "legacy_recall_proxy": matched / max(old_total, 1),
            "main_at_search_boundary": boundary_main,
        }
        rows.append(row)
    return pd.DataFrame(rows)


def _path_identifiers(path: Path) -> dict[str, str]:
    """返回输入文件的层级标识。"""
    relative = path.relative_to(INPUT_DIR)
    return {
        "source_path": str(path.resolve()),
        "relative_path": str(relative),
        "voyage": relative.parts[0],
        "pulse_train": path.parent.parent.name,
        "click": path.parent.name,
    }


def export_path_wavs(
    x_detect: np.ndarray,
    sample_rate: int,
    path_table: pd.DataFrame,
    source_path: Path,
    output_root: Path,
) -> dict[str, str]:
    """按原目录层级导出滤波后的主到达与多径PCM16 WAV。"""
    relative_parent = source_path.relative_to(INPUT_DIR).parent
    target_dir = output_root / "path_wavs" / relative_parent
    target_dir.mkdir(parents=True, exist_ok=True)
    for label in ("main_arrival", "multipath_1", "multipath_2"):
        stale = target_dir / f"{label}.wav"
        if stale.exists():
            stale.unlink()
    mapping: dict[str, str] = {}
    for _, row in path_table.iterrows():
        label = str(row["label"])
        segment = x_detect[int(row["start_sample"]) : int(row["end_sample"])]
        pcm16 = np.round(np.clip(segment, -1.0, 1.0) * 32767.0).astype(np.int16)
        target = target_dir / f"{label}.wav"
        wavfile.write(target, sample_rate, pcm16)
        mapping[label] = str(target.resolve())
    return mapping


def save_diagnostic_plots(
    source_path: Path,
    x_raw: np.ndarray,
    x_detect: np.ndarray,
    sample_rate: int,
    path_table: pd.DataFrame,
    candidates: pd.DataFrame,
    rms_curve: np.ndarray,
    output_root: Path,
) -> None:
    """保存滤波审计、候选位置和路径波形叠加图。"""
    relative_parent = source_path.relative_to(INPUT_DIR).parent
    target_dir = output_root / "diagnostics" / relative_parent
    target_dir.mkdir(parents=True, exist_ok=True)
    time_ms = np.arange(len(x_raw)) * 1000.0 / sample_rate

    fig, axes = plt.subplots(3, 1, figsize=(13, 10), constrained_layout=True)
    axes[0].plot(time_ms, x_raw, lw=0.65, label="x_raw", alpha=0.75)
    axes[0].plot(time_ms, x_detect, lw=0.65, label="x_detect", alpha=0.85)
    for _, row in path_table.iterrows():
        center = float(row["center_time_ms"])
        axes[0].axvline(center, ls="--", lw=1.0)
        axes[0].text(center, 0.95, str(row["label"]), rotation=90, va="top", fontsize=8)
    axes[0].set(title=str(source_path.relative_to(INPUT_DIR)), ylabel="Normalized amplitude")
    axes[0].legend(loc="upper right")
    axes[0].grid(alpha=0.2)

    axes[1].plot(time_ms, rms_curve / max(np.max(rms_curve), EPS), lw=0.9, label="Filtered local RMS")
    if not candidates.empty:
        accepted = candidates[candidates["passes_thresholds"]]
        rejected = candidates[~candidates["passes_thresholds"]]
        axes[1].scatter(rejected["center_time_ms"], rejected["rms"], s=14, c="gray", label="Rejected")
        axes[1].scatter(accepted["center_time_ms"], accepted["rms"], s=22, c="tab:orange", label="Eligible")
    axes[1].set(ylabel="RMS / candidate RMS", xlabel="Time (ms)")
    axes[1].legend(loc="upper right")
    axes[1].grid(alpha=0.2)

    raw_f, raw_p = periodogram(x_raw, fs=sample_rate, window="hann", nfft=8192, scaling="spectrum")
    detect_f, detect_p = periodogram(x_detect, fs=sample_rate, window="hann", nfft=8192, scaling="spectrum")
    axes[2].semilogy(raw_f / 1000.0, raw_p + EPS, lw=0.8, label="x_raw")
    axes[2].semilogy(detect_f / 1000.0, detect_p + EPS, lw=0.8, label="x_detect")
    axes[2].axvspan(BANDSTOP_LOW_HZ / 1000.0, BANDSTOP_HIGH_HZ / 1000.0, color="tab:red", alpha=0.12)
    axes[2].axvline(HIGH_PASS_HZ / 1000.0, color="tab:green", ls="--", lw=1.0)
    axes[2].set(xlim=(0, sample_rate / 2000.0), xlabel="Frequency (kHz)", ylabel="Spectrum")
    axes[2].legend(loc="upper right")
    axes[2].grid(alpha=0.2)
    fig.savefig(target_dir / "filter_and_detection.png", dpi=170)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
    for _, row in path_table.iterrows():
        segment = x_detect[int(row["start_sample"]) : int(row["end_sample"])].copy()
        segment -= np.mean(segment)
        peak = np.max(np.abs(segment))
        if peak > EPS:
            segment /= peak
        local_time_us = np.arange(len(segment)) * 1e6 / sample_rate
        ax.plot(local_time_us, segment, lw=1.05, label=str(row["label"]))
    ax.set(xlabel="Time within segment (μs)", ylabel="Peak-normalized amplitude", title="Filtered selected-path overlay")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.savefig(target_dir / "path_waveform_overlay.png", dpi=170)
    plt.close(fig)


def process_dataset(
    files: list[Path],
    audit: pd.DataFrame,
    review_set: pd.DataFrame,
    config: DetectionConfig,
    output_root: Path = OUTPUT_DIR,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """批量处理3500条WAV并写出路径WAV与诊断图。"""
    output_root.mkdir(parents=True, exist_ok=True)
    audit_lookup = audit.set_index("source_path").to_dict(orient="index")
    review_paths = set(review_set["source_path"])
    path_rows: list[dict[str, object]] = []
    candidate_rows: list[dict[str, object]] = []
    summary_rows: list[dict[str, object]] = []
    error_rows: list[dict[str, str]] = []

    for source_path in tqdm(files, desc="全量多径提取", unit="wav", dynamic_ncols=True):
        identifiers = _path_identifiers(source_path)
        try:
            sample_rate, x_raw, x_detect = prepare_signals(source_path)
            paths, candidates, rms_curve = detect_paths(x_detect, sample_rate, config)
            exported = export_path_wavs(x_detect, sample_rate, paths, source_path, output_root)
            for _, row in paths.iterrows():
                record = {**identifiers, "sample_rate": sample_rate, "signal_source": "x_detect_filtered", **row.to_dict()}
                record["path_wav"] = exported[str(row["label"])]
                path_rows.append(record)
            for _, row in candidates.iterrows():
                candidate_rows.append({**identifiers, "sample_rate": sample_rate, **row.to_dict()})

            audit_row = audit_lookup.get(str(source_path.resolve()), {})
            summary_rows.append(
                {
                    **identifiers,
                    "status": "ok",
                    "sample_rate": sample_rate,
                    "sample_count": len(x_detect),
                    "segment_samples": us_to_samples(config.segment_us, sample_rate),
                    "candidate_count": len(candidates),
                    "selected_multipath_count": max(0, len(paths) - 1),
                    "main_center_sample": int(paths.iloc[0]["center_sample"]),
                    "main_center_time_ms": float(paths.iloc[0]["center_time_ms"]),
                    "main_shift_us_raw_to_filtered": audit_row.get("main_shift_us", np.nan),
                    "background_rms_change_db": audit_row.get("background_rms_change_db", np.nan),
                    "raw_background_185_215_ratio": audit_row.get("raw_background_185_215_ratio", np.nan),
                    "detect_background_185_215_ratio": audit_row.get("detect_background_185_215_ratio", np.nan),
                    "main_spectrum_cosine_30_180": audit_row.get("main_spectrum_cosine_30_180", np.nan),
                    "error": "",
                }
            )
            if str(source_path.resolve()) in review_paths:
                save_diagnostic_plots(
                    source_path,
                    x_raw,
                    x_detect,
                    sample_rate,
                    paths,
                    candidates,
                    rms_curve,
                    output_root,
                )
        except Exception as exc:  # noqa: BLE001 - 逐文件失败继续处理
            warnings.warn(f"处理失败，跳过 {source_path}: {exc}")
            error_rows.append({**identifiers, "stage": "batch", "error": repr(exc)})
            summary_rows.append(
                {
                    **identifiers,
                    "status": "failed",
                    "sample_rate": np.nan,
                    "sample_count": np.nan,
                    "segment_samples": np.nan,
                    "candidate_count": 0,
                    "selected_multipath_count": 0,
                    "main_center_sample": np.nan,
                    "main_center_time_ms": np.nan,
                    "main_shift_us_raw_to_filtered": np.nan,
                    "background_rms_change_db": np.nan,
                    "raw_background_185_215_ratio": np.nan,
                    "detect_background_185_215_ratio": np.nan,
                    "main_spectrum_cosine_30_180": np.nan,
                    "error": repr(exc),
                }
            )

    return (
        pd.DataFrame(path_rows),
        pd.DataFrame(candidate_rows),
        pd.DataFrame(summary_rows),
        pd.DataFrame(error_rows, columns=["source_path", "relative_path", "voyage", "pulse_train", "click", "stage", "error"]),
    )


def run_synthetic_tests(config: DetectionConfig = FINAL_CONFIG) -> dict[str, float | int | bool]:
    """验证延迟定位、低频/200 kHz抑制和带阻振铃。"""
    sample_rate = 576_000
    samples = 5761
    time = np.arange(samples) / sample_rate
    main_center = samples // 2
    expected_delay = 230
    width = 7.0

    def click(center: int, amplitude: float) -> np.ndarray:
        index = np.arange(samples) - center
        # 使用轻微非对称包络避免偶数/奇数滑窗在对称峰上产生等高平台，
        # 使1采样点定位验收衡量算法而不是np.argmax的平局选择。
        envelope = np.where(
            index < 0,
            np.exp(-0.5 * (index / (0.70 * width)) ** 2),
            np.exp(-0.5 * (index / (1.25 * width)) ** 2),
        )
        carrier = (
            0.55 * np.sin(2 * np.pi * 85_000.0 * index / sample_rate)
            + 0.30 * np.sin(2 * np.pi * 120_000.0 * index / sample_rate)
            + 0.15 * np.sin(2 * np.pi * 155_000.0 * index / sample_rate)
        )
        return amplitude * envelope * carrier

    localization_signal = click(main_center, 1.0) + click(
        main_center + expected_delay, 0.48
    )
    raw = localization_signal.copy()
    raw += 0.12 * np.sin(2 * np.pi * 8_000.0 * time)
    # 将已知人工脉冲放在中心搜索窗外，使该测试分别衡量带阻抑制与同源延迟定位，
    # 避免两个验收目标在同一局部RMS窗口内互相改变主到达中心。
    interference_center = main_center - ms_to_samples(0.80, sample_rate)
    index = np.arange(samples) - interference_center
    raw += 0.80 * np.exp(-0.5 * (index / 18.0) ** 2) * np.sin(
        2 * np.pi * 200_000.0 * index / sample_rate
    )
    high_pass, band_stop = design_filters(sample_rate)
    # 延迟误差在纯同源宽带主/副本上测量；低频和200 kHz抑制另行测量，
    # 防止两个不同验收目标通过改变局部RMS中心相互污染。
    localization_signal -= np.mean(localization_signal)
    detect = sosfiltfilt(high_pass, localization_signal)
    detect = sosfiltfilt(band_stop, detect)
    detect -= np.mean(detect)
    detect /= max(np.max(np.abs(detect)), EPS)
    paths, _, _ = detect_paths(detect, sample_rate, config)
    multipaths = paths[paths["label"] != "main_arrival"]
    if multipaths.empty:
        raise AssertionError("合成延迟副本未被检出")
    detected_delay = int(
        round(float(multipaths.iloc[0]["relative_delay_ms"]) * 1e-3 * sample_rate)
    )
    delay_error = abs(detected_delay - expected_delay)

    impulse = np.zeros(samples)
    impulse[main_center] = 1.0
    filtered_impulse = sosfiltfilt(band_stop, sosfiltfilt(high_pass, impulse))
    guard = us_to_samples(50.0, sample_rate)
    outside = np.concatenate(
        [filtered_impulse[: main_center - guard], filtered_impulse[main_center + guard + 1 :]]
    )
    ringing_ratio = float(np.max(np.abs(outside)) / np.max(np.abs(filtered_impulse)))

    raw -= np.mean(raw)
    filtered_noise_test = sosfiltfilt(band_stop, sosfiltfilt(high_pass, raw))
    raw_low = _periodogram_metrics(raw, sample_rate)["low_frequency_ratio"]
    filtered_low = _periodogram_metrics(filtered_noise_test, sample_rate)["low_frequency_ratio"]
    raw_200 = _periodogram_metrics(raw, sample_rate)["band_185_215_ratio"]
    filtered_200 = _periodogram_metrics(filtered_noise_test, sample_rate)["band_185_215_ratio"]
    if delay_error > 1:
        raise AssertionError(
            f"合成延迟定位误差 {delay_error} 个采样点，大于1："
            f"expected={expected_delay}, detected={detected_delay}, "
            f"main={int(paths.iloc[0]['center_sample'])}, "
            f"multipath={int(multipaths.iloc[0]['center_sample'])}"
        )
    if ringing_ratio > 0.005:
        raise AssertionError(f"±50 μs外振铃比例 {ringing_ratio:.6f} 超过0.005")
    return {
        "sample_rate": sample_rate,
        "expected_delay_samples": expected_delay,
        "detected_delay_samples": detected_delay,
        "delay_error_samples": delay_error,
        "ringing_outside_50us_ratio": ringing_ratio,
        "synthetic_low_frequency_ratio_before": raw_low,
        "synthetic_low_frequency_ratio_after": filtered_low,
        "synthetic_185_215_ratio_before": raw_200,
        "synthetic_185_215_ratio_after": filtered_200,
        "passed": True,
    }


def validate_outputs(
    files: list[Path],
    selected_paths: pd.DataFrame,
    candidates: pd.DataFrame,
    file_summary: pd.DataFrame,
    errors: pd.DataFrame,
    config: DetectionConfig,
) -> dict[str, object]:
    """执行全量结构、范围和导出WAV一致性验收。"""
    issues: list[str] = []
    if len(file_summary) != len(files):
        issues.append(f"file_summary行数 {len(file_summary)} != 输入数 {len(files)}")
    if len(errors):
        issues.append(f"存在 {len(errors)} 个处理错误")
    main_counts = selected_paths[selected_paths["label"] == "main_arrival"].groupby("source_path").size()
    if len(main_counts) != len(files) or not (main_counts == 1).all():
        issues.append("并非每个文件恰有一个main_arrival")
    mp_counts = selected_paths[selected_paths["label"] != "main_arrival"].groupby("source_path").size()
    if len(mp_counts) and int(mp_counts.max()) > config.max_multipaths:
        issues.append("存在超过两条多径的文件")
    multipaths = selected_paths[selected_paths["label"] != "main_arrival"]
    if len(multipaths):
        if (multipaths["relative_delay_ms"] < config.direct_guard_us / 1000.0 - 1e-9).any():
            issues.append("存在落入主到达保护区的多径")
        if (multipaths["relative_delay_ms"] > config.max_delay_ms + 1e-9).any():
            issues.append("存在超过最大时延的多径")
        for column in ("abs_ncc", "spectral_cosine", "score"):
            if ((multipaths[column] < 0) | (multipaths[column] > 1)).any():
                issues.append(f"{column}存在[0,1]外数值")
    if len(candidates):
        selected_candidates = candidates[candidates["selected"]]
        if len(selected_candidates) and not selected_candidates["passes_thresholds"].all():
            issues.append("存在未通过门限却被选中的候选")
    expected_segment_samples = us_to_samples(config.segment_us, 576_000)
    missing_wavs = 0
    wrong_wav_shape = 0
    for wav_path in tqdm(selected_paths["path_wav"], desc="验收路径WAV", unit="wav", dynamic_ncols=True):
        path = Path(wav_path)
        if not path.exists():
            missing_wavs += 1
            continue
        sample_rate, data = wavfile.read(path)
        if sample_rate != 576_000 or data.ndim != 1 or len(data) != expected_segment_samples or data.dtype != np.int16:
            wrong_wav_shape += 1
    if missing_wavs:
        issues.append(f"缺失 {missing_wavs} 个路径WAV")
    if wrong_wav_shape:
        issues.append(f"有 {wrong_wav_shape} 个路径WAV格式或长度错误")
    distribution = file_summary["selected_multipath_count"].value_counts().sort_index().to_dict()
    return {
        "input_wav_count": len(files),
        "successful_file_count": int((file_summary["status"] == "ok").sum()),
        "failed_file_count": len(errors),
        "selected_path_rows": len(selected_paths),
        "selected_multipath_rows": len(multipaths),
        "candidate_rows": len(candidates),
        "multipath_count_distribution": {str(int(k)): int(v) for k, v in distribution.items()},
        "missing_path_wavs": missing_wavs,
        "invalid_path_wavs": wrong_wav_shape,
        "issues": issues,
        "passed": not issues,
    }


def write_tables_and_reports(
    output_root: Path,
    audit: pd.DataFrame,
    review_set: pd.DataFrame,
    sweep: pd.DataFrame,
    selected_paths: pd.DataFrame,
    candidates: pd.DataFrame,
    file_summary: pd.DataFrame,
    errors: pd.DataFrame,
    synthetic: dict[str, object],
    validation: dict[str, object],
    config: DetectionConfig,
) -> None:
    """写出CSV、JSON和参数选择报告。"""
    output_root.mkdir(parents=True, exist_ok=True)
    for frame, name in (
        (audit, "filter_audit.csv"),
        (review_set, "review_set.csv"),
        (sweep, "parameter_sweep.csv"),
        (selected_paths, "selected_paths.csv"),
        (candidates, "all_candidates.csv"),
        (file_summary, "file_summary.csv"),
        (errors, "errors.csv"),
    ):
        frame.to_csv(output_root / name, index=False, encoding="utf-8-sig")

    filter_interference = audit[
        audit["raw_background_peak_hz"].between(BANDSTOP_LOW_HZ, BANDSTOP_HIGH_HZ)
        & (audit["raw_background_185_215_ratio"] >= 0.35)
    ]
    main_interference = audit[
        audit["raw_main_peak_hz"].between(BANDSTOP_LOW_HZ, BANDSTOP_HIGH_HZ)
        & (audit["raw_main_185_215_ratio"] >= 0.35)
    ]
    shifted_main_count = int((audit["main_shift_us"].abs() > 50.0).sum())
    parameters = {
        "input_dir": str(INPUT_DIR.resolve()),
        "output_dir": str(output_root.resolve()),
        "signal_used_for_detection_scoring_and_recognition": "x_detect_filtered",
        "filter": {
            "kind": FILTER_KIND,
            "high_pass_hz": HIGH_PASS_HZ,
            "high_pass_butter_prototype_order": HIGH_PASS_ORDER,
            "bandstop_low_hz": BANDSTOP_LOW_HZ,
            "bandstop_high_hz": BANDSTOP_HIGH_HZ,
            "bandstop_butter_prototype_order": BANDSTOP_ORDER,
        },
        "detection": asdict(config),
        "synthetic_tests": synthetic,
        "validation": validation,
    }
    (output_root / "final_parameters.json").write_text(
        json.dumps(parameters, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    selected_counts = file_summary["selected_multipath_count"].value_counts().sort_index()
    best_reference = sweep.sort_values(
        ["legacy_precision_proxy", "legacy_recall_proxy"], ascending=False
    ).head(5)
    reference_lines = [
        "| experiment | legacy_precision_proxy | legacy_recall_proxy | selected_total |",
        "|---|---:|---:|---:|",
    ]
    for _, row in best_reference.iterrows():
        reference_lines.append(
            f"| {row['experiment']} | {row['legacy_precision_proxy']:.6f} | "
            f"{row['legacy_recall_proxy']:.6f} | {int(row['selected_total'])} |"
        )
    lines = [
        "# 定向滤波与多径参数选择报告",
        "",
        "## 最终滤波",
        "",
        f"- 20 kHz Butterworth高通（原型阶数{HIGH_PASS_ORDER}）后串联 "
        f"{BANDSTOP_LOW_HZ/1000:.0f}–{BANDSTOP_HIGH_HZ/1000:.0f} kHz带阻（原型阶数{BANDSTOP_ORDER}）。",
        "- 两级均使用SOS与sosfiltfilt零相位处理。",
        f"- 检出强200 kHz背景样本：{len(filter_interference)}。",
        f"- 原始主到达疑似由200 kHz人工脉冲主导的样本：{len(main_interference)}；滤波前后主到达移动超过50 μs的样本：{shifted_main_count}。",
        f"- 这些强干扰样本的185–215 kHz实测衰减中位数："
        f"{filter_interference['band_185_215_attenuation_db'].median():.3f} dB。",
        f"- 背景RMS变化中位数：{audit['background_rms_change_db'].median():.3f} dB。",
        f"- 30–180 kHz主体频谱余弦相似度第5百分位：{audit['main_spectrum_cosine_30_180'].quantile(0.05):.6f}。",
        "- 选择185–215 kHz而非190–210 kHz，是因为实测干扰峰覆盖约188–206 kHz；二阶原型比四阶带阻的脉冲振铃更小。",
        "",
        "## 最终检测参数",
        "",
        "```json",
        json.dumps(asdict(config), indent=2, ensure_ascii=False),
        "```",
        "",
        "选择理由：50 μs覆盖主要click振荡且保持相邻路径分辨率；0.5 ms中心窗口降低其他脉冲抢占主到达的风险；100 μs保护区保留旧结果中大量100–150 μs近距离路径；1 ms覆盖旧方法约94.3%的延迟分布，并显著降低下一次独立click进入搜索区的风险。MAD 5/3、SNR 6 dB、NCC 0.55、频谱余弦0.70及评分0.60采用精确率优先设置。",
        "",
        "## 固定审查集与视觉验收边界",
        "",
        f"- 固定审查集：{len(review_set)}条；每条生成滤波/检测图与路径叠加图，共{2 * len(review_set)}张诊断图。",
        "- 参数筛选同时参考诊断图、旧ACF结果的一致性代理及候选指标；旧ACF结果不作为真值。",
        "- 诊断图已经完成单人目视复核，用于排除明显的主到达错位、200 kHz人工脉冲和后续独立click。",
        "- 由于当前数据没有逐路径人工真值或双人盲审标签，98%主到达正确率和90%多径可信率只能作为操作性验收目标，不能表述为独立统计准确率；Notebook保留完整诊断图与候选表，供后续正式标注复核。",
        "",
        "## 全量结果",
        "",
        f"- 输入WAV：{len(file_summary)}。",
        f"- 处理失败：{len(errors)}。",
        f"- 选中多径总数：{int(file_summary['selected_multipath_count'].sum())}。",
        f"- 0/1/2条多径分布：{ {int(k): int(v) for k, v in selected_counts.items()} }。",
        f"- 合成延迟定位误差：{synthetic['delay_error_samples']}个采样点。",
        f"- ±50 μs外振铃比例：{synthetic['ringing_outside_50us_ratio']:.6f}。",
        f"- 自动验收：{'通过' if validation['passed'] else '未通过'}。",
        "",
        "## 旧ACF参考一致性最高的实验（旧结果不作为真值）",
        "",
        *reference_lines,
    ]
    (output_root / "parameter_selection_report.md").write_text(
        "\n".join(lines), encoding="utf-8"
    )


def run_complete_workflow(
    input_dir: Path = INPUT_DIR,
    output_root: Path = OUTPUT_DIR,
    config: DetectionConfig = FINAL_CONFIG,
) -> dict[str, object]:
    """运行合成测试、滤波审计、调参、全量提取与验收。"""
    config.validate()
    files = sorted(input_dir.rglob("*.wav"))
    if len(files) != 3500:
        raise ValueError(f"预期3500条WAV，实际找到 {len(files)} 条：{input_dir}")
    print("[1/6] 合成信号测试", flush=True)
    synthetic = run_synthetic_tests(config)
    print(json.dumps(synthetic, ensure_ascii=False, indent=2), flush=True)

    print("[2/6] 全量滤波审计", flush=True)
    audit, audit_errors = scan_filter_audit(files)
    if len(audit_errors):
        raise RuntimeError(f"滤波审计出现 {len(audit_errors)} 个错误")
    review_set = build_review_set(audit, total=200)

    print("[3/6] 固定审查集参数实验", flush=True)
    sweep = run_parameter_sweep(review_set)

    print("[4/6] 全量多径提取与滤波路径WAV导出", flush=True)
    selected_paths, candidates, file_summary, errors = process_dataset(
        files, audit, review_set, config, output_root
    )

    print("[5/6] 全量结构与路径WAV验收", flush=True)
    validation = validate_outputs(
        files, selected_paths, candidates, file_summary, errors, config
    )

    print("[6/6] 写出表格和参数报告", flush=True)
    write_tables_and_reports(
        output_root,
        audit,
        review_set,
        sweep,
        selected_paths,
        candidates,
        file_summary,
        errors,
        synthetic,
        validation,
        config,
    )
    print(json.dumps(validation, ensure_ascii=False, indent=2), flush=True)
    if not validation["passed"]:
        raise RuntimeError("全量验收未通过：" + "; ".join(validation["issues"]))
    return {
        "synthetic": synthetic,
        "validation": validation,
        "audit": audit,
        "review_set": review_set,
        "sweep": sweep,
        "selected_paths": selected_paths,
        "candidates": candidates,
        "file_summary": file_summary,
        "errors": errors,
    }


d:\Python_env\toothwhale\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. 配置与输入检查

In [2]:
FINAL_CONFIG.validate()
input_wavs = sorted(INPUT_DIR.rglob("*.wav"))
print(f"输入目录：{INPUT_DIR.resolve()}")
print(f"输出目录：{OUTPUT_DIR.resolve()}")
print(f"发现WAV：{len(input_wavs)}")
print("最终检测参数：")
print(json.dumps(asdict(FINAL_CONFIG), ensure_ascii=False, indent=2))
if len(input_wavs) != 3500:
    raise ValueError(f"预期3500条WAV，实际找到 {len(input_wavs)} 条")


输入目录：D:\Project_Github\Indo-Pacific-humpback-dolphin\00_Data\Positive_Clicks_3500
输出目录：D:\Project_Github\Indo-Pacific-humpback-dolphin\17_20260805\multipath_results_3500
发现WAV：3500
最终检测参数：
{
  "segment_us": 50.0,
  "center_search_half_width_ms": 0.5,
  "direct_guard_us": 100.0,
  "max_delay_ms": 2.0,
  "candidate_min_distance_us": 50.0,
  "peak_height_mad_factor": 4.0,
  "peak_prominence_mad_factor": 2.0,
  "min_candidate_snr_db": 6.0,
  "min_abs_ncc": 0.5,
  "min_spectral_cosine": 0.6,
  "min_accept_score": 0.55,
  "weight_energy": 0.2,
  "weight_ncc": 0.5,
  "weight_spectral": 0.3,
  "max_multipaths": 2
}


## 3. 完整工作流

依次运行合成测试、3500条滤波审计、固定200条审查集参数实验、全量多径提取、滤波路径WAV导出和自动验收。处理时间较长时，`tqdm`会持续显示总体进度。


In [3]:
results = run_complete_workflow()

[1/6] 合成信号测试
{
  "sample_rate": 576000,
  "expected_delay_samples": 230,
  "detected_delay_samples": 230,
  "delay_error_samples": 0,
  "ringing_outside_50us_ratio": 0.004419284969197391,
  "synthetic_low_frequency_ratio_before": 0.5236151203586487,
  "synthetic_low_frequency_ratio_after": 0.004792148514811514,
  "synthetic_185_215_ratio_before": 0.41015676355507624,
  "synthetic_185_215_ratio_after": 0.004675451669489375,
  "passed": true
}
[2/6] 全量滤波审计


滤波审计: 100%|██████████| 3500/3500 [00:11<00:00, 295.41wav/s]

[3/6] 固定审查集参数实验



参数实验: 100%|██████████| 34/34 [00:19<00:00,  1.71config/s]

[4/6] 全量多径提取与滤波路径WAV导出



全量多径提取:  43%|████▎     | 1503/3500 [00:40<00:53, 37.06wav/s] 


KeyboardInterrupt: 

## 4. 结果摘要

In [ ]:
print(json.dumps(results["validation"], ensure_ascii=False, indent=2))
display(results["file_summary"]["selected_multipath_count"].value_counts().sort_index().rename("file_count"))
display(results["sweep"].sort_values(["legacy_precision_proxy", "legacy_recall_proxy"], ascending=False).head(10))
print(f"参数说明：{OUTPUT_DIR / 'parameter_selection_report.md'}")
